# A-nanTa V7.3.1.4 — Training-Matched Preflight / NVIDIA L4 Smoke Test

**Purpose:** Gate the full-model training notebook **before** the 140-epoch run. Mirrors the actual configuration, causal features, labels, complete 10,041,978-parameter architecture, all eight production stage objectives, stage-specific trainable parameter selection and joint PCGrad. Does **not** train for full epochs or modify the production notebook, caches, or checkpoints.

**Run:** Colab → Runtime → Change runtime type → NVIDIA L4 GPU → Run all.

**Three independent gates:**
1. **Contract** — compare exact code-cell hashes with current `main` training notebook. Stop if out of sync.
2. **Real-data prerequisite** — read-only inspection of cached BTC/ETH/SOL and NSX files; records `NOT_READY` when missing, never downloads 5 years accidentally.
3. **L4 execution** — controlled synthetic-batch Forward / Backward / AdamW steps in all A–H stages, plus validation and isolated checkpoint state restore. Times are **per small diagnostic batch**, not full-epoch estimates.

No synthetic results should be described as real-data model accuracy or training-readiness on their own. Final verdict requires **all three** gates.


In [ ]:
# 0 — Training-source provenance (fail closed on divergence)
import os,json,hashlib,urllib.request,platform,datetime,time,gc,math,random,traceback,sys,zipfile,tempfile
from pathlib import Path
MAIN_RAW_URL='https://raw.githubusercontent.com/ekjsnskkd-byte/A-nanTa/main/notebooks/training/A-nanTa_V7314_TRAINING_QUALITY_GATES.ipynb'
EXPECTED_SOURCE_HASHES={'1': '91af82726010a171331aa8844ce07eb908487f00bb88875509c364fc326f635f', '2': '7c860588a7d85d344d28a3ae49f2984e3787435c8d422d06186e5282ad1129c3', '4': 'c9860ba09564b841d3b32c1e8be05b516fdcbf03d1ded6062f2a8d5105002afe', '5': '5270008a958887ca6be9da8c8ba986f87429da2d3897bdeec251aebb494c4630', '10': '0555dc9ab7b1e90314a41fa59528fe7d4aaa3ba5d1e584d9d90ee509ea8fac42', '11': 'a56cb830c7b9cd2da0cff4d93702c32f4a5ffd1bd91fd6c198b0039f12992800'}
REPORT_DIR=Path('/content/ananta_v7314_preflight') if Path('/content').exists() else Path.cwd()/'ananta_v7314_preflight'
REPORT_DIR.mkdir(parents=True,exist_ok=True)
PROVENANCE_STATUS='NOT_CHECKED'
try:
    with urllib.request.urlopen(MAIN_RAW_URL,timeout=25) as response:
        live=json.load(response)
    current={str(k):hashlib.sha256(''.join(live['cells'][k]['source']).encode()).hexdigest() for k in map(int,EXPECTED_SOURCE_HASHES)}
    differences={i: {'expected':v,'actual':current.get(i)} for i,v in EXPECTED_SOURCE_HASHES.items() if current.get(i)!=v}
    if differences:
        PROVENANCE_STATUS='MISMATCH'
        raise RuntimeError('Training notebook changed! Refresh smoke test from current main: '+str(differences))
    PROVENANCE_STATUS='MATCH'
except Exception as e:
    if PROVENANCE_STATUS!='MISMATCH':
        PROVENANCE_STATUS='UNVERIFIED'
    raise RuntimeError('Cannot verify training source; fail closed to prevent stale smoke results') from e
print('Training-source code match:',PROVENANCE_STATUS)


## Original model environment, configuration, feature and label functions (verbatim from training notebook)

In [ ]:
# Cell 1 — dependencies and reproducibility
import os, json, math, random, hashlib
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
try:
    import pyarrow  # parquet reader
except ImportError:
    import sys, subprocess
    if 'google.colab' not in sys.modules:
        raise ImportError('pyarrow is required: pip install pyarrow')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
    import pyarrow
import torch
from torch import nn
from torch.nn import functional as F
SEED=2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:',torch.__version__,'device:',DEVICE)
# Optimized precision: NVIDIA L4 supports BF16 tensor cores; no GradScaler needed for BF16.
USE_BF16=(DEVICE.type=='cuda' and torch.cuda.is_bf16_supported())
from contextlib import nullcontext
def amp_context():
    return torch.autocast(device_type='cuda',dtype=torch.bfloat16) if USE_BF16 else nullcontext()
print('BF16 autocast enabled:',USE_BF16)


In [ ]:
# Config — set DATA_DIR to your mounted Google Drive Parquet folder.
CFG={
 'data_dir':'/content/manticore_v730_crypto_m5',
 'data_glob':None,'use_google_drive':True,'assets':['BTCUSD','ETHUSDT','SOLUSDT'],
 'pool_assets':['BTCUSD','ETHUSDT','SOLUSDT'],'target_asset':'BTCUSD','context_assets':['NSXUSD'],
 'cache_dir':'/content/manticore_v730_pool_finetune_cache','output_dir':'/content/drive/MyDrive/HERMES_Colab/ai_training/manticore_v70_trend_specialist_v2/v7314_training_quality_checkpoints',
 'timestamp_semantics':'open','bar_minutes':5,'horizons':[1,3,12,48],'lookback':128,'purge_bars':192,
  'binance_market':'futures/cm','binance_symbol':'BTCUSD_PERP',
  'binance_contract_usd':100.0,'binance_auto_download':True,
 'binance_aux_market':'futures/um','binance_aux_symbols':{'ETHUSDT':'ETHUSDT','SOLUSDT':'SOLUSDT'},
 'binance_aux_archive_base':'/content/drive/MyDrive/HERMES_Colab/ai_training/binance_vision/um_pooled_aggTrades',
 'binance_aux_m5_base':'/content/drive/MyDrive/HERMES_Colab/ai_training/historical_market_cache/binance_um_pooled_aggtrades',
 'histdata_nsx_zip_dir':'/content/drive/MyDrive/HERMES_Colab/ai_training/histdata_tick/NSXUSD',
 'histdata_nsx_m5_cache':'/content/drive/MyDrive/HERMES_Colab/ai_training/historical_market_cache/histdata_nsxusd_tick/NSXUSD_M5.parquet',
 'dollar_bar_target_usd':100000.0,
 'verify_binance_sha256':True, 'checkpoint_every_steps':250,
 'validation_report_months':2, 'data_pipeline_version':'v7313_4',
 'trailing_feature_window':200,
  'binance_start':'2020-10','binance_end':None,
  'binance_archive_dir':'/content/drive/MyDrive/HERMES_Colab/ai_training/binance_vision/cm_BTCUSD_PERP_aggTrades',
  'tick_bar_cache':'/content/drive/MyDrive/HERMES_Colab/ai_training/historical_market_cache/binance_cm_btcusd_aggtrades/BTCUSD_M5.parquet',
 'train_years':5,'val_months':6,'cal_months':3,'sealed_months':3,
 'recent_half_life_days':270.,'recent_uniform_floor':0.15,
 'stress_boost':2.0,'stress_vol_quantile':0.995,'stress_cap':3.0,
 'batch_size':64,'epochs':110,'patience':5,'lr':3e-4,
 'finetune_lr_multiplier':0.15,
 'finetune_epochs':{'A_SSL':3,'B_REGIME':3,'C_GRAPH':3,'D_FORECAST':4,'E_PATH':4,'F_EXPERTS':4,'G_DECISION':3,'H_JOINT':6},
 'graph_history_points':8,'graph_history_stride':4,'path_rank':4,'path_knots':[1,2,3,4,6,8,12,16,24,32,40,48],
 'mtf_timeframes':{'M15':'15min','H1':'1h','H4':'4h','D1':'1D'},'mtf_window':32,'mtf_encoder_layers':2,
 'hidden':192,'transformer_layers':5,'attention_heads':6,'gru_layers':2,'graph_layers':2,'expert_width':512,'experts':8,'top_k':2,'graph_neighbors':6,'loader_workers':2,'prefetch_factor':2,'persistent_workers':True,'graph_index_cache':True,
  'use_bf16':True,
 'neutral_move_bps':5.,'label_vol_multiplier':.25,
 'barrier_vol_multiplier':1.,'reference_cost_bps':3.,
 'min_rows':700,'min_train_sequences':100,
 'max_train_steps_per_epoch':None,  # optional bounded research runs; no silent caps
 'direction_weight':1.,'decision_weight':1.,'quantile_weight':1.,
 'rv_weight':.5,'excursion_weight':.5,'barrier_weight':.5,'regime_weight':.25,
 'route_balance_weight':.02,'pattern_weight':.35,'jump_weight':.25,'path_weight':.35,'graph_weight':1.,
 'mtf_aux_weight':.15,
 'stage_epochs':{'A_SSL':20,'B_REGIME':10,'C_GRAPH':10,'D_FORECAST':20,'E_PATH':15,'F_EXPERTS':15,'G_DECISION':10,'H_JOINT':10},
 'stage_lr_mult':{'A_SSL':1.,'B_REGIME':.65,'C_GRAPH':.5,'D_FORECAST':.5,'E_PATH':.35,'F_EXPERTS':.35,'G_DECISION':.5,'H_JOINT':.15},
 'stage_patience':{'A_SSL':5,'B_REGIME':4,'C_GRAPH':4,'D_FORECAST':6,'E_PATH':5,'F_EXPERTS':5,'G_DECISION':4,'H_JOINT':5},
 'stage_max_val_steps':None, # None means full validation
 'research_adverse_penalty':0.20, 'utility_temperature':0.35,
 'path_nll_weight':.20,'action_q_weight':.35,'pattern_detail_weight':.25,
 'use_pcgrad_joint':True,'expert_val_batches':3,
 'validate_expert_ablation':True,'diagnostic_rolling_windows':4,
 'fine_tune_guard_relative_tolerance':0.08, 'diagnostic_gradient_assert':True,
 'run_sealed_test':False,  # explicit one-time review after freeze
}
assert 3<=CFG['train_years']<=5 and CFG['recent_half_life_days']>0
assert 0<CFG['recent_uniform_floor']<1 and CFG['stress_boost']>=1
assert max(CFG['horizons'])<CFG['purge_bars']

assert set(CFG['finetune_epochs'])==set(CFG['stage_epochs']) and all(e>0 for e in CFG['finetune_epochs'].values())
assert 0<CFG['finetune_lr_multiplier']<=1 and CFG['dollar_bar_target_usd']>0
assert CFG['assets']==CFG['pool_assets']==['BTCUSD','ETHUSDT','SOLUSDT'], 'Pooled crypto asset contract'
assert CFG['target_asset']=='BTCUSD' and CFG['context_assets']==['NSXUSD']
assert CFG['path_knots'][-1] == max(CFG['horizons']) and sorted(set(CFG['path_knots'])) == CFG['path_knots']
# User switch: keep FP32 when BF16 is disabled even on L4.
USE_BF16=bool(USE_BF16 and CFG.get('use_bf16',True))


In [ ]:
# Cell 3 — validate data and calculate causal features
FEATURES=['ret1','ret3','ret12','vol12','vol48','vol_ratio','efficiency','range_norm','position','downside','skew24','kurt24','entropy24','spread_rel','spread_present','volume_log','volume_present','trend_strength_96','volatility_har_12','volatility_har_48','volatility_har_288','hurst_proxy','wavelet_band','jump_proxy','mean_reversion_z','autocorr24','taker_dollar_imbalance','taker_buy_ratio','notional_log','trade_count_log','imbalance_ewm','flow_shock','dollar_event_count_log','dollar_event_imbalance','dollar_event_activity']
# Asset identity is intentionally NOT used by the model: new assets use the same causal statistics.
def load_asset_file(path):
    """Read verified local-mounted Drive files; no external downloads and no CSV conversion."""
    path=Path(path)
    if path.suffix.lower()=='.parquet':
        d=pd.read_parquet(path)
    elif path.suffix.lower()=='.csv':
        d=pd.read_csv(path)
    else:
        raise ValueError(f'Unsupported market file: {path}')
    d.columns=[str(c).lower().strip() for c in d.columns]
    if 'timestamp' not in d.columns:
        for key in ('datetime','date_time','time','date'):
            if key in d.columns:
                d=d.rename(columns={key:'timestamp'});break
    if 'timestamp' not in d.columns and isinstance(d.index,pd.DatetimeIndex):
        d=d.reset_index().rename(columns={d.index.name or 'index':'timestamp'})
    if 'timestamp' in d.columns and pd.api.types.is_numeric_dtype(d['timestamp']):
        raise ValueError(f'{path}: numeric epoch timestamps need explicit unit (s/ms/ns); aborting rather than guessing')
    missing={'timestamp','open','high','low','close'}-set(d.columns)
    if missing:raise ValueError(f'{path}: missing {sorted(missing)}')
    d['timestamp']=pd.to_datetime(d['timestamp'],utc=True,errors='raise')
    if d['timestamp'].isna().any() or d['timestamp'].duplicated().any():
        raise ValueError(f'{path}: null/duplicate timestamps; fix raw source, do not silently deduplicate')
    d=d.sort_values('timestamp').set_index('timestamp')
    for k in ('open','high','low','close','bid','ask','volume','notional_usd','signed_notional_usd','buy_notional_usd','trade_count'):
        if k in d:d[k]=pd.to_numeric(d[k],errors='coerce')
    if d[['open','high','low','close']].isna().any().any():raise ValueError(f'{path}: missing prices')
    if len(d)<CFG['min_rows']:raise ValueError(f'{path}: insufficient bars {len(d)}')
    if ((d[['open','high','low','close']]<=0).any().any() or (d.high<d[['open','close','low']].max(axis=1)).any()
            or (d.low>d[['open','close','high']].min(axis=1)).any()):
        raise ValueError(f'{path}: invalid OHLC')
    if {'bid','ask'}<=set(d.columns) and ((d.ask<d.bid)&d.ask.notna()&d.bid.notna()).any():
        raise ValueError(f'{path}: crossed quotes')
    # Non-uniform timestamps are allowed, but the report must disclose cadence/gaps per asset.
    return d

def fast_permutation_entropy(log_returns,window=24):
    """Exact causal order-3 permutation entropy over each 24-return window;
    rolling sample contains window-2 consecutive triplets. O(N), six permutations.
    """
    a=np.asarray(log_returns,dtype=np.float64)
    x=a[:-2];y=a[1:-1];z=a[2:]
    # stable ties: earlier temporal element wins
    rank_x=((y<x).astype('int8')+(z<x).astype('int8'))
    rank_y=((x<=y).astype('int8')+(z<y).astype('int8'))
    rank_z=3-rank_x-rank_y
    # Each permutation identified by base-3 encoded ranks, six distinct values.
    patterns=np.full(len(a),-1,dtype=np.int16)
    good=np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
    patterns[2:][good]=(rank_x[good]*9+rank_y[good]*3+rank_z[good]).astype(np.int16)
    out=np.full(len(a),np.nan,dtype=np.float64)
    count=np.zeros(len(a),dtype=np.int16)
    ent=np.zeros(len(a),dtype=np.float64)
    # 22 consecutive patterns in a 24-return window, causal at index t.
    for key in (5,7,11,15,19,21):
        hit=pd.Series((patterns==key).astype('int8'))
        count_arr=hit.rolling(window-2,min_periods=window-2).sum().to_numpy()
        count+=np.nan_to_num(count_arr,nan=0).astype(np.int16)
        prob=np.nan_to_num(count_arr,nan=0)/(window-2)
        ent-=np.where(prob>0,prob*np.log(np.maximum(prob,1e-30)),0)
    # exclude any missing triplets in rolling window
    valid=pd.Series((patterns>=0).astype('int8')).rolling(window-2,min_periods=window-2).sum().to_numpy()==window-2
    out[valid]=ent[valid]/np.log(6)
    return out

def _causal_features_one_contiguous_segment(d):
    p=d.close;lr=np.log(p).diff();v=lr.rolling(48,min_periods=24).std()
    f=pd.DataFrame(index=d.index)
    f['ret1']=lr;f['ret3']=np.log(p/p.shift(3));f['ret12']=np.log(p/p.shift(12))
    f['vol12']=lr.rolling(12,min_periods=12).std();f['vol48']=v
    f['vol_ratio']=f.vol12/(v+1e-8)
    f['efficiency']=p.diff(24).abs()/(p.diff().abs().rolling(24).sum()+1e-8)
    f['range_norm']=(d.high-d.low)/(p*v.replace(0,np.nan)+1e-8)
    f['position']=(p-p.rolling(24).min())/(p.rolling(24).max()-p.rolling(24).min()+1e-8)
    f['downside']=lr.clip(upper=0).pow(2).rolling(24).mean().pow(.5)
    f['skew24']=lr.rolling(24).skew();f['kurt24']=lr.rolling(24).kurt()
    f['entropy24']=fast_permutation_entropy(lr.to_numpy(),24)
    if {'bid','ask'}<=set(d.columns):
        raw=((d.ask-d.bid)/p).where((d.ask>=d.bid)&(d.bid>0))
        f['spread_rel']=raw;f['spread_present']=raw.notna().astype(float)
    else:f['spread_rel']=np.nan;f['spread_present']=0.
    if 'volume' in d:
        x=np.log1p(d.volume.clip(lower=0));f['volume_log']=x-x.rolling(48,min_periods=24).mean()
        f['volume_present']=d.volume.notna().astype(float)
    else:f['volume_log']=np.nan;f['volume_present']=0.
    # Causal multiscale and distributional Quant state features (no future data).
    f['trend_strength_96']=np.log(p/p.shift(96))/(lr.rolling(96,min_periods=48).std()*np.sqrt(96)+1e-9)
    f['volatility_har_12']=lr.pow(2).rolling(12,min_periods=12).mean()
    f['volatility_har_48']=lr.pow(2).rolling(48,min_periods=48).mean()
    f['volatility_har_288']=lr.pow(2).rolling(288,min_periods=144).mean()
    f['hurst_proxy']=np.log((lr.rolling(48,min_periods=24).sum().abs()+1e-9)/
                   (lr.abs().rolling(48,min_periods=24).sum()+1e-9))
    f['wavelet_band']=lr.rolling(8,min_periods=8).mean()-lr.rolling(48,min_periods=24).mean()
    f['jump_proxy']=(lr.abs()/(lr.abs().rolling(48,min_periods=24).median()+1e-9)).clip(upper=50)
    f['mean_reversion_z']=(np.log(p)-np.log(p).rolling(48,min_periods=24).mean())/(v*np.sqrt(48)+1e-8)
    f['autocorr24']=lr.rolling(24,min_periods=24).corr(lr.shift(1))
    # Aggregated executions from Binance COIN-M Futures. 100 USD/contract for BTCUSD_PERP;
    # is_buyer_maker=True means aggressive SELL. These are observed trade flows, not liquidations.
    notional=d['notional_usd'] if 'notional_usd' in d else pd.Series(np.nan,index=d.index)
    signed=d['signed_notional_usd'] if 'signed_notional_usd' in d else pd.Series(np.nan,index=d.index)
    buys=d['buy_notional_usd'] if 'buy_notional_usd' in d else pd.Series(np.nan,index=d.index)
    trades=d['trade_count'] if 'trade_count' in d else pd.Series(np.nan,index=d.index)
    imbalance=(signed/(notional+1e-9)).where(notional>0)
    f['taker_dollar_imbalance']=imbalance
    f['taker_buy_ratio']=(buys/(notional+1e-9)).where(notional>0)
    f['notional_log']=np.log1p(notional.clip(lower=0))-np.log1p(notional.clip(lower=0)).rolling(48,min_periods=24).mean()
    f['trade_count_log']=np.log1p(trades.clip(lower=0))-np.log1p(trades.clip(lower=0)).rolling(48,min_periods=24).mean()
    f['imbalance_ewm']=imbalance.ewm(span=24,adjust=False,min_periods=12).mean()
    f['flow_shock']=(signed.abs()/(signed.abs().rolling(48,min_periods=24).median()+1e-9)).clip(0,50)
    # Dollar-imbalance events: threshold-crossing of executed USD notional, aligned to CLOSED M5 bar.
    # Each event was completed no later than this bar, and no future quote is used.
    events=d['dollar_event_count'] if 'dollar_event_count' in d else pd.Series(np.nan,index=d.index)
    signed_events=d['dollar_event_imbalance'] if 'dollar_event_imbalance' in d else pd.Series(np.nan,index=d.index)
    f['dollar_event_count_log']=np.log1p(events.clip(lower=0))
    f['dollar_event_imbalance']=signed_events
    f['dollar_event_activity']=events.clip(lower=0)/(events.clip(lower=0).rolling(48,min_periods=24).mean()+1e-9)
    return f.replace([np.inf,-np.inf],np.nan)

def causal_features(d):
    """Causal features must not bridge missing source candles or market closures."""
    if d.empty:return _causal_features_one_contiguous_segment(d)
    times=d.index.to_numpy(dtype='datetime64[ns]')
    if len(times)<2:return _causal_features_one_contiguous_segment(d)
    steps=np.diff(times).astype('timedelta64[ns]').astype(np.int64)
    if np.any(steps<=0):raise ValueError('Non-monotonic feature source timestamps')
    expected=int(np.min(steps)) # 5m on source bars; e.g. 15m/1h on resampled inputs.
    boundaries=np.r_[0,np.where(steps!=expected)[0]+1,len(d)]
    if len(boundaries)==2:return _causal_features_one_contiguous_segment(d)
    parts=[_causal_features_one_contiguous_segment(d.iloc[a:b]) for a,b in zip(boundaries[:-1],boundaries[1:])]
    return pd.concat(parts).reindex(d.index)


In [ ]:
# Cell 4 — future-only supervised targets. Inputs never use these columns.
# Regime targets are CAUSAL heuristic teachers (trend persistence, volatility, transition).
# Decision teacher uses future FIRST-TOUCH barrier utility, NOT direction classification.
def build_labels(d,horizons):
    lr=np.log(d.close);r=lr.diff();y=pd.DataFrame(index=d.index)
    local_vol=r.rolling(48,min_periods=24).std().clip(lower=1e-8)
    eff=(d.close.diff(24).abs()/(d.close.diff().abs().rolling(24).sum()+1e-8)).fillna(0)
    vratio=(r.rolling(12,min_periods=12).std()/(local_vol+1e-8)).fillna(1)
    y['regime_trend']=eff.clip(0,1) # an explicit causal weak teacher, not ground truth
    y['regime_high_vol']=(vratio>1.5).astype(float)
    y['regime_transition']=((eff-eff.shift(12)).abs()>.2).astype(float)
    squared=r.pow(2)
    for h in horizons:
        future=lr.shift(-h)-lr
        y[f'ret_{h}']=future
        threshold=np.maximum(CFG['neutral_move_bps']*1e-4,CFG['label_vol_multiplier']*local_vol*np.sqrt(h))
        y[f'dir_{h}']=np.where(future>threshold,2,np.where(future< -threshold,0,1)).astype(float)
        y.loc[future.isna(),f'dir_{h}']=np.nan
        # Require COMPLETE future window. max/min must not be computed on an incomplete tail.
        base_close=d.close.to_numpy(dtype=np.float64)
        hi=d.high.to_numpy(dtype=np.float64);lo=d.low.to_numpy(dtype=np.float64)
        max_exc=np.full(len(d),-np.inf);min_exc=np.full(len(d),np.inf)
        valid=np.ones(len(d),dtype=bool)
        upper=(CFG['barrier_vol_multiplier']*local_vol*np.sqrt(h)).clip(lower=CFG['neutral_move_bps']*1e-4).to_numpy()
        up=base_close*np.exp(upper);dn=base_close*np.exp(-upper)
        barrier=np.ones(len(d),dtype=np.int8) # 0 SHORT, 1 WAIT, 2 LONG
        undecided=np.isfinite(upper)&(upper>=CFG['reference_cost_bps']*1e-4)
        ambiguous=np.zeros(len(d),dtype=bool)
        for j in range(1,h+1):
            fhi=np.full(len(d),np.nan);flo=np.full(len(d),np.nan)
            fhi[:-j]=hi[j:];flo[:-j]=lo[j:]
            valid &= np.isfinite(fhi)&np.isfinite(flo)
            with np.errstate(divide='ignore',invalid='ignore'):
                max_exc=np.maximum(max_exc,np.log(fhi/base_close))
                min_exc=np.minimum(min_exc,np.log(flo/base_close))
            both=(fhi>=up)&(flo<=dn)&undecided
            only_up=(fhi>=up)&~both&undecided
            only_dn=(flo<=dn)&~both&undecided
            ambiguous |= both
            barrier[only_up]=2;barrier[only_dn]=0
            undecided[only_up|only_dn|both]=False
        y[f'mfe_{h}']=np.where(valid,max_exc,np.nan)
        y[f'mae_{h}']=np.where(valid,min_exc,np.nan)
        # Future realized variance: squared future close returns, no future values in inputs
        y[f'rv_{h}']=sum(squared.shift(-j) for j in range(1,h+1))
        y[f'barrier_{h}']=np.where(ambiguous,np.nan,barrier.astype(float))
        # Same-bar high/low cannot establish execution order; exclude this supervised sample.
        # Event outcomes: first-touch breakout, reversal and jump within horizon.
        past_high=d.high.rolling(48,min_periods=24).max().shift(1).to_numpy(dtype=np.float64)
        past_low=d.low.rolling(48,min_periods=24).min().shift(1).to_numpy(dtype=np.float64)
        # Future maximum/minimum log excursions were computed only for targets.
        y[f'pattern_{h}']=np.where(np.isfinite(past_high)&np.isfinite(past_low),
            ((base_close*np.exp(max_exc)>past_high)|(base_close*np.exp(min_exc)<past_low)).astype(float),np.nan)
        y[f'jump_{h}']=np.where(valid,((max_exc-min_exc)>3.5*np.maximum(local_vol.to_numpy()*np.sqrt(h),1e-8)).astype(float),np.nan)
        # Coarse path-shape target: final displacement relative to maximum signed excursion.
        y[f'path_{h}']=np.where(valid,np.clip(future.to_numpy()/(np.maximum(max_exc-min_exc,1e-8)),-2,2),np.nan)
        y.iloc[-h:,y.columns.get_loc(f'barrier_{h}')]=np.nan
        # A row count is NOT a time horizon when source M5 is missing.
        # Monotonic, unique timestamps plus exact elapsed time prove every 5-minute bar exists.
        delta=np.timedelta64(h*CFG['bar_minutes'],'m')
        time_utc=d.index.to_numpy(dtype='datetime64[ns]')
        connected=np.zeros(len(d),dtype=bool)
        connected[:-h]=(time_utc[h:]-time_utc[:-h])==delta
        for suffix in ('ret','dir','mfe','mae','rv','barrier','pattern','jump','path'):
            y.loc[~connected,f'{suffix}_{h}']=np.nan

    # LONG/SHORT/WAIT decision teacher uses first-touch event at longest horizon.
    # This is a noisy research target, NOT an executable trading policy or broker simulation.
    y['decision']=y[f'barrier_{horizons[-1]}']
    # Event pattern targets (future-only): false breakout, trend reversal, trend continuation.
    past_mom=np.log(d.close/d.close.shift(24)).to_numpy(dtype=np.float64)
    for h in horizons:
        complete=y[f'ret_{h}'].notna().to_numpy()
        ph=d.high.rolling(48,min_periods=24).max().shift(1).to_numpy(dtype=np.float64)
        pl=d.low.rolling(48,min_periods=24).min().shift(1).to_numpy(dtype=np.float64)
        terminal=d.close.shift(-h).to_numpy(dtype=np.float64)
        mfe=y[f'mfe_{h}'].to_numpy(dtype=np.float64)
        mae=y[f'mae_{h}'].to_numpy(dtype=np.float64)
        valid=complete&np.isfinite(ph)&np.isfinite(pl)&np.isfinite(past_mom)
        touched=((d.close.to_numpy()*np.exp(mfe)>ph)|(d.close.to_numpy()*np.exp(mae)<pl))
        inside=(terminal<=ph)&(terminal>=pl)
        y[f'false_breakout_{h}']=np.where(valid,(touched&inside).astype(float),np.nan)
        # Volatility-normalized patterns. No subjective chart labels.
        future=y[f'ret_{h}'].to_numpy(dtype=np.float64)
        move_cut=(CFG['label_vol_multiplier']*local_vol*np.sqrt(h)).to_numpy(dtype=np.float64)
        past_signal=np.sign(past_mom)
        real_move=np.abs(future)>np.maximum(move_cut,CFG['neutral_move_bps']*1e-4)
        y[f'reversal_{h}']=np.where(valid,(real_move&(future*past_signal<0)).astype(float),np.nan)
        y[f'continuation_{h}']=np.where(valid,(real_move&(future*past_signal>0)).astype(float),np.nan)
        time_utc=d.index.to_numpy(dtype='datetime64[ns]')
        connected=np.zeros(len(d),dtype=bool)
        connected[:-h]=(time_utc[h:]-time_utc[:-h])==np.timedelta64(h*CFG['bar_minutes'],'m')
        for suffix in ('false_breakout','reversal','continuation'):
            y.loc[~connected,f'{suffix}_{h}']=np.nan

    # Stochastic path is trained on log-return INCREMENTS at fixed future knots.
    previous=0
    for k in CFG['path_knots']:
        y[f'path_increment_{k}']=np.log(d.close.shift(-k)/d.close.shift(-previous))
        time_utc=d.index.to_numpy(dtype='datetime64[ns]')
        connected=np.zeros(len(d),dtype=bool)
        connected[:-k]=(time_utc[k:]-time_utc[:-k])==np.timedelta64(k*CFG['bar_minutes'],'m')
        y.loc[~connected,f'path_increment_{k}']=np.nan
        previous=k
    return y


In [ ]:
# 1 — Laboratory isolation: source CFG is unchanged except lab-only output destination.
import time,gc,tempfile,datetime
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from contextlib import nullcontext
LAB_ROOT=REPORT_DIR
PRODUCTION_OUTPUT=CFG['output_dir']
CFG['output_dir']=str(LAB_ROOT/'checkpoints')
# Keep original model/loader hyperparameters. Only the test batch count is bounded.
BATCH_SIZE=CFG['batch_size']  # full batch 64 matches training; reduce only with explicit user override
DIAGNOSTIC_BATCH_SIZE=int(os.environ.get('ANANTA_SMOKE_BATCH_SIZE',str(BATCH_SIZE)))
assert 1 <= DIAGNOSTIC_BATCH_SIZE <= BATCH_SIZE
TEST_STEPS_PER_STAGE=int(os.environ.get('ANANTA_SMOKE_STEPS','1'))
assert 1 <= TEST_STEPS_PER_STAGE <= 4
EXPECTED_GPU='L4'
ALLOW_NON_L4=os.environ.get('ANANTA_SMOKE_CPU_DEBUG','0')=='1'
if not ALLOW_NON_L4:
    assert torch.cuda.is_available(), 'Select NVIDIA L4 in Colab Runtime before running smoke test'
    assert 'L4' in torch.cuda.get_device_name(0).upper(), f'Expected NVIDIA L4, got {torch.cuda.get_device_name(0)!r}'
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU DEBUG ONLY')
print('BF16 active:',USE_BF16,'Production output untouched:',PRODUCTION_OUTPUT)


In [ ]:
# 2 — Read-only production data readiness gate. No mass download or write to Drive.
from datetime import datetime, timezone
DATA_PATHS={
 'BTCUSD':Path(CFG['tick_bar_cache']),
 'ETHUSDT':Path(CFG['binance_aux_m5_base'])/'ETHUSDT_M5.parquet',
 'SOLUSDT':Path(CFG['binance_aux_m5_base'])/'SOLUSDT_M5.parquet',
 'NSXUSD':Path(CFG['histdata_nsx_m5_cache']),
}
try:
    from google.colab import drive
    if CFG.get('use_google_drive',False) and Path('/content').exists() and not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive',force_remount=False)
except ImportError:
    pass
except Exception as e:
    print('Drive mount warning:',repr(e))

DATA_AUDIT={}
try:
    import pyarrow.parquet as pq
except ImportError:
    pq=None
for asset,path in DATA_PATHS.items():
    item={'path':str(path),'exists':path.is_file()}
    if path.is_file():
        item['bytes']=path.stat().st_size
        try:
            if pq is None:raise RuntimeError('pyarrow unavailable')
            pf=pq.ParquetFile(path)
            item['rows']=pf.metadata.num_rows
            item['row_groups']=pf.metadata.num_row_groups
            item['columns']=list(pf.schema.names)
            required={'open','high','low','close'}
            item['schema_ok']=required.issubset(set(item['columns']))
            item['readable']=pf.metadata.num_rows>CFG['min_rows'] and item['schema_ok']
            # Probe bounded final row group only to catch unreadable files. May be a large group.
            if pf.metadata.num_row_groups and pf.metadata.row_group(pf.metadata.num_row_groups-1).num_rows<=250000:
                probe=pf.read_row_group(pf.metadata.num_row_groups-1,columns=['close']).to_pandas()
                assert len(probe)>0 and np.isfinite(probe.close.to_numpy(dtype=float)).all()
                item['final_row_group_probe']='PASS'
            else:item['final_row_group_probe']='SKIPPED_LARGE_ROW_GROUP'
        except Exception as e:
            item['readable']=False;item['error']=str(e)
    else:item['readable']=False
    DATA_AUDIT[asset]=item
    print(asset, {k:v for k,v in item.items() if k!='columns'})
DATA_READY=all(x['readable'] for x in DATA_AUDIT.values())
print('Data cache available:',DATA_READY,'(not proof of coverage or full history)')


## Exact production model and objective functions (not rewritten proxies)

In [ ]:
# 3 — Synthetic scale initialization follows the SAME train-only median formulas as production.
# Real production scales come from actual train targets in main Cell 8; no claim of identical values here.
H=CFG['horizons'];lb=CFG['lookback'];TF_NAMES=list(CFG['mtf_timeframes'])
_rng=np.random.default_rng(2026)
_fake_train_returns=_rng.normal(0,.002,size=(4096,len(H)))
_fake_train_variance=np.abs(_rng.normal(1e-5,2e-6,size=(4096,)))
ret_scale=max(float(np.median(np.abs(_fake_train_returns[:,len(H)-1]))),1e-5)
rv_scale=max(float(np.median(_fake_train_variance)),1e-8)
print('Synthetic diagnostic train-only scales:',ret_scale,rv_scale)


In [ ]:
# Full probabilistic architecture — causal sequence + evolving graph + direct bypass + stochastic paths
class CausalDiagonalStateSpace(nn.Module):
    """Learnable stable diagonal linear recurrence using parallel causal convolution.
    Kernel k_j=(1-alpha)*alpha**j; all outputs at t use inputs <=t only.
    """
    def __init__(self,dim):
        super().__init__()
        self.decay_logits=nn.Parameter(torch.linspace(.5,5.,dim))
        self.in_projection=nn.Linear(dim,dim)
        self.out_gate=nn.Linear(dim,dim)
        self.norm=nn.LayerNorm(dim)
    def forward(self,h):
        B,T,D=h.shape
        alpha=torch.sigmoid(self.decay_logits).clamp(.05,.999)
        lags=torch.arange(T,device=h.device,dtype=h.dtype)
        impulse=(1-alpha[:,None])*alpha[:,None].pow(lags[None,:])
        states=F.conv1d(self.in_projection(h).transpose(1,2),
                        impulse.flip(-1).unsqueeze(1),padding=T-1,groups=D)[...,:T]
        return self.norm(states.transpose(1,2)*torch.sigmoid(self.out_gate(h)))

class DynamicTemporalGraph(nn.Module):
    def __init__(self,dim,heads=6,layers=2):
        super().__init__()
        self.layers=nn.ModuleList([nn.MultiheadAttention(dim,heads,batch_first=True) for _ in range(layers)])
        self.norms=nn.ModuleList([nn.LayerNorm(dim) for _ in range(layers)])
        self.temporal=nn.GRU(dim,dim,batch_first=True)
        self.fuse=nn.Sequential(nn.Linear(dim*2,dim),nn.GELU(),nn.LayerNorm(dim))
    def forward(self,center,center_history,nodes,mask):
        # [B,N,T,D] per-time dynamic neighbor attention; every snapshot is <= center anchor.
        B,N,T,D=nodes.shape
        assert center_history.shape==(B,T,D), 'Graph center states must align with graph snapshots'
        center_repeat=center_history.reshape(B*T,1,D)
        z=nodes.permute(0,2,1,3).reshape(B*T,N,D)
        valid=mask.permute(0,2,1).reshape(B*T,N)
        tokens=torch.cat([center_repeat,z],dim=1)
        padding=torch.cat([torch.zeros((B*T,1),dtype=torch.bool,device=center.device),~valid],dim=1)
        for attn,norm in zip(self.layers,self.norms):
            upd,_=attn(tokens,tokens,tokens,key_padding_mask=padding,need_weights=False)
            tokens=norm(tokens+upd)
        seq=tokens[:,0].reshape(B,T,D)
        state=self.temporal(seq)[0][:,-1]
        any_neighbor=mask.any((1,2))
        fused=self.fuse(torch.cat([center,state],dim=-1))
        return torch.where(any_neighbor[:,None],fused,center)

class MultiTimeframeFusion(nn.Module):
    """Independent causal M15/H1/H4/D1 encoders plus mask-aware attention to M5.
    Historical windows end at the last CLOSED bar available at the M5 decision time.
    Missing timeframes have zero attention probability, not merely zero-filled inputs.
    """
    def __init__(self,features,hidden,heads,layers):
        super().__init__()
        self.encoders=nn.ModuleList([nn.TransformerEncoder(
            nn.TransformerEncoderLayer(hidden,heads,hidden*2,dropout=.1,batch_first=True,norm_first=True),
            layers,enable_nested_tensor=False) for _ in TF_NAMES])
        self.input=nn.ModuleList([nn.Sequential(nn.Linear(features,hidden),nn.LayerNorm(hidden)) for _ in TF_NAMES])
        self.tokens=nn.Parameter(torch.randn(len(TF_NAMES),hidden)*.02)
        self.query=nn.Linear(hidden,hidden)
        self.key=nn.Linear(hidden,hidden)
        self.value=nn.Linear(hidden,hidden)
        self.output=nn.Sequential(nn.Linear(hidden*2,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.norm=nn.LayerNorm(hidden)
        self.scale=hidden**-.5
    def forward(self,query,mtf,mtf_mask):
        B,K,W,Fdim=mtf.shape
        assert K==len(TF_NAMES) and mtf_mask.shape==(B,K,W)
        tokens=[];present=[]
        causal=torch.triu(torch.ones(W,W,device=mtf.device,dtype=torch.bool),diagonal=1)
        for j in range(K):
            good=mtf_mask[:,j,:].bool()
            # all-zero samples must still produce finite outputs; output suppressed by present mask.
            h=self.input[j](mtf[:,j])+self.tokens[j]
            h=self.encoders[j](h,mask=causal)
            idx=good.long().sum(1).sub(1).clamp(min=0)
            # Padding on left: the last candle is always W-1 if this timeframe exists.
            token=h[:,-1]
            valid=good.any(1)
            tokens.append(token);present.append(valid)
        t=torch.stack(tokens,dim=1)
        available=torch.stack(present,dim=1)
        scores=(self.query(query)[:,None,:]*self.key(t)).sum(-1)*self.scale
        scores=scores.masked_fill(~available,-1e9)
        alpha=F.softmax(scores,dim=-1)*available.float()
        alpha=alpha/alpha.sum(-1,keepdim=True).clamp(min=1e-8)
        fused=(alpha.unsqueeze(-1)*self.value(t)).sum(1)
        blended=self.output(torch.cat([query,fused],dim=-1))
        return torch.where(available.any(1,keepdim=True),blended,query),alpha

class UniversalQuantModel(nn.Module):
    def __init__(self,features,hidden=192,nexperts=8,topk=2,nh=4,
                 transformer_layers=5,attention_heads=6,gru_layers=2,graph_layers=2,expert_width=512):
        super().__init__()
        self.nh=nh;self.topk=topk;self.npath=len(CFG['path_knots']);self.rank=CFG['path_rank']
        self.input=nn.Sequential(nn.Linear(features,hidden),nn.LayerNorm(hidden))
        self.conv=nn.Conv1d(hidden,hidden,5,padding=4,groups=hidden)
        self.encoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(hidden,attention_heads,hidden*4,
            dropout=.1,batch_first=True,norm_first=True),transformer_layers,enable_nested_tensor=False)
        self.longrange=nn.GRU(hidden,hidden,num_layers=gru_layers,batch_first=True)
        self.ssm=CausalDiagonalStateSpace(hidden)
        self.graph_input=nn.Sequential(nn.Linear(features,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.graph=DynamicTemporalGraph(hidden,attention_heads,graph_layers)
        self.graph_fusion=nn.Sequential(nn.Linear(hidden*4,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.mtf=MultiTimeframeFusion(features,hidden,attention_heads,CFG['mtf_encoder_layers'])
        self.ssl_predict=nn.Linear(hidden,features)
        self.regime=nn.Sequential(nn.Linear(hidden,hidden),nn.GELU(),nn.Linear(hidden,7))
        self.router=nn.Sequential(nn.Linear(hidden+7,hidden),nn.GELU(),nn.Linear(hidden,nexperts))
        self.experts=nn.ModuleList([nn.Sequential(nn.Linear(hidden,expert_width),nn.GELU(),
                           nn.Linear(expert_width,expert_width),nn.GELU(),nn.Linear(expert_width,hidden))
                           for _ in range(nexperts)])
        # Direct connections from original market context, graph and regime; no mandatory MoE bottleneck.
        self.forecast_fusion=nn.Sequential(nn.Linear(hidden*3+7,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.direction=nn.Linear(hidden,nh*3)
        self.quantiles=nn.Linear(hidden,nh*3)
        self.rv=nn.Linear(hidden,nh)
        self.excursions=nn.Linear(hidden,nh*2)
        self.barrier=nn.Linear(hidden,nh*3)
        self.pattern=nn.Linear(hidden,nh)
        self.pattern_detail=nn.Linear(hidden,nh*3)
        self.jump=nn.Linear(hidden,nh)
        self.path=nn.Linear(hidden,nh) # legacy coarse shape, retained for continuity
        self.path_distribution=nn.Linear(hidden,self.npath*(2+self.rank))
        self.decision_quantiles=nn.Linear(hidden+nh*3,3*3)  # action utility q10/q50/q90
        self.decision=nn.Sequential(nn.Linear(hidden+nh*3+9,hidden),nn.GELU(),nn.Linear(hidden,3))
    def forward(self,x,neighbors=None,neighbor_mask=None,use_experts=True,ablated_expert=None,
                use_graph=True,mtf=None,mtf_mask=None,use_mtf=True):
        B,T,_=x.shape
        h=self.input(x)
        h=h+self.conv(h.transpose(1,2))[...,:T].transpose(1,2)
        causal=torch.triu(torch.ones(T,T,dtype=torch.bool,device=x.device),diagonal=1)
        temporal_states=self.encoder(h,mask=causal)
        temporal=temporal_states[:,-1]
        recurrent=self.longrange(h)[0][:,-1]
        state_space=self.ssm(h)[:,-1]
        graph_repr=temporal
        if use_graph and neighbors is not None and neighbors.shape[1]>0:
            assert neighbors.ndim==4 and neighbor_mask.ndim==3
            nodes=self.graph_input(neighbors)
            gp=neighbors.shape[2];stride=CFG['graph_history_stride']
            history=temporal_states[:,-1-(gp-1)*stride::stride]
            if history.shape[1]!=gp:raise ValueError('Lookback shorter than graph history span')
            graph_repr=self.graph(temporal,history,nodes,neighbor_mask.bool())
        z0=self.graph_fusion(torch.cat([temporal,recurrent,state_space,graph_repr],-1))
        if use_mtf and mtf is not None:
            assert mtf_mask is not None
            z0,mtf_attention=self.mtf(z0,mtf,mtf_mask)
        else:
            mtf_attention=torch.zeros(B,len(TF_NAMES),device=x.device,dtype=x.dtype)
        regime=self.regime(z0)
        route=self.router(torch.cat([z0,torch.sigmoid(regime)],-1))
        top=route.topk(min(self.topk,route.shape[-1]),dim=-1)
        gate=torch.zeros_like(route).scatter(1,top.indices,F.softmax(top.values.float(),dim=-1).to(route.dtype))
        if ablated_expert is not None:
            gate=gate.clone();gate[:,int(ablated_expert)]=0.
            gate=gate/gate.sum(1,keepdim=True).clamp(min=1e-8)
        # Sparse dispatch: only run experts selected for each sample.
        mixture=torch.zeros_like(z0)
        if use_experts:
            for j,expert in enumerate(self.experts):
                selected=torch.nonzero(gate[:,j]>0,as_tuple=True)[0]
                if selected.numel():
                    expert_contribution=expert(z0.index_select(0,selected))*gate[selected,j,None]
                    # Under BF16 autocast, expert output may differ from FP32 mixture dtype.
                    # index_add requires matching dtypes; this conversion retains gradients.
                    mixture=mixture.index_add(0,selected,expert_contribution.to(dtype=mixture.dtype))
        z=z0+mixture
        ctx=z+self.forecast_fusion(torch.cat([z,z0,graph_repr,torch.sigmoid(regime)],dim=-1))
        direction=self.direction(ctx).reshape(B,self.nh,3)
        raw=self.quantiles(ctx).reshape(B,self.nh,3)
        median=raw[...,1]*ret_scale
        quantiles=torch.stack([median-F.softplus(raw[...,0])*ret_scale,median,
                               median+F.softplus(raw[...,2])*ret_scale],dim=-1)
        raw_path=self.path_distribution(ctx).reshape(B,self.npath,2+self.rank)
        path_mu=raw_path[...,0]*ret_scale
        path_diag=(F.softplus(raw_path[...,1]).clamp(max=10.)+.03)*ret_scale
        path_factors=torch.tanh(raw_path[...,2:])*ret_scale
        direction_probs=F.softmax(direction,dim=-1).reshape(B,-1)
        dq=self.decision_quantiles(torch.cat([ctx,direction_probs],dim=-1)).reshape(B,3,3)
        utility_median=dq[...,1]
        action_quantiles=torch.stack([utility_median-F.softplus(dq[...,0]),utility_median,
                                    utility_median+F.softplus(dq[...,2])],dim=-1)*ret_scale
        decision=self.decision(torch.cat([ctx,direction_probs,action_quantiles.reshape(B,-1)/ret_scale],-1))
        return {'direction':direction,'quantiles':quantiles,'rv':F.softplus(self.rv(ctx))*rv_scale,
                'excursions':self.excursions(ctx).reshape(-1,self.nh,2)*ret_scale,
                'barrier':self.barrier(ctx).reshape(-1,self.nh,3),'pattern':self.pattern(ctx),
                'pattern_detail':self.pattern_detail(ctx).reshape(-1,self.nh,3),
                'jump':self.jump(ctx),'path':self.path(ctx),
                'path_mu':path_mu,'path_diag':path_diag,'path_factors':path_factors,
                'action_quantiles':action_quantiles,'decision':decision,'regime':regime,
                'ssl_prediction':self.ssl_predict(z0),'gate':gate,
                'router_probs':F.softmax(route,dim=-1),'dispatch':(gate>0).float(),
                'mtf_attention':mtf_attention}

def path_distribution_nll(o,target):
    # NVIDIA L4: CUDA autocast may downcast the internal matrix multiply to BF16.
    # LowRankMultivariateNormal then calls cuSOLVER Cholesky, which rejects BF16.
    # Disable autocast for the ENTIRE distribution construction/log_prob,
    # not just the input casts; gradients still flow back to the BF16 model.
    with torch.autocast(device_type=o['path_mu'].device.type, enabled=False):
        scale=float(ret_scale)
        mu=o['path_mu'].float()/scale
        factors=o['path_factors'].float()/scale
        diagonal=(o['path_diag'].float()/scale).square().clamp(min=1e-5)
        dist=torch.distributions.LowRankMultivariateNormal(
            loc=mu, cov_factor=factors, cov_diag=diagonal)
        nll=-dist.log_prob(target.float()/scale).mean()
    return nll

@torch.no_grad()
def path_forecast_quantiles(o):
    # True path distribution: cumulative correlated Gaussian increments at future knots.
    mu=o['path_mu'].cumsum(-1)
    diag_var=o['path_diag'].square().cumsum(-1)
    factor_sum=o['path_factors'].cumsum(1)
    var=diag_var+factor_sum.square().sum(-1)
    sd=var.clamp(min=1e-15).sqrt()
    return torch.stack([mu-1.2815515655*sd,mu,mu+1.2815515655*sd],dim=-1)

@torch.no_grad()
def sample_correlated_price_paths(o,starting_price,scenarios=64):
    # Draw correlated log-return increments in FP32 even if caller uses CUDA BF16 autocast.
    with torch.autocast(device_type=o['path_mu'].device.type, enabled=False):
        dist=torch.distributions.LowRankMultivariateNormal(
            loc=o['path_mu'].float(),cov_factor=o['path_factors'].float(),
            cov_diag=o['path_diag'].float().square().clamp(min=1e-14))
        increments=dist.sample((int(scenarios),)) # [scenarios, batch, path_knots]
        return float(starting_price)*increments.cumsum(-1).exp()

model=UniversalQuantModel(len(FEATURES),CFG['hidden'],CFG['experts'],CFG['top_k'],len(H),
    CFG['transformer_layers'],CFG['attention_heads'],CFG['gru_layers'],CFG['graph_layers'],CFG['expert_width']).to(DEVICE)
PARAMETERS=sum(p.numel() for p in model.parameters())
print('FULL probabilistic trainable parameters:',PARAMETERS)


In [ ]:
def pinball(pred,y):
    tau=torch.tensor([.1,.5,.9],dtype=pred.dtype,device=pred.device)
    error=y.unsqueeze(-1)-pred
    return torch.maximum(tau*error,(tau-1)*error).mean()


def cost_aware_utilities(r,m,n):
    # Research-only, future teacher; NEVER available to model during inference.
    terminal=r[:,-1]
    long=terminal-CFG['reference_cost_bps']*1e-4-CFG['research_adverse_penalty']*(-n[:,-1]).clamp(min=0)
    short=-terminal-CFG['reference_cost_bps']*1e-4-CFG['research_adverse_penalty']*m[:,-1].clamp(min=0)
    return torch.stack([short,torch.zeros_like(terminal),long],dim=-1)


def regime_loss(o,b):
    target=b[7];mask=b[13]
    assert target.shape[1]==7 and mask.shape==target.shape
    err=F.binary_cross_entropy_with_logits(o['regime'],target,reduction='none')
    return (err*mask).sum()/mask.sum().clamp(min=1)


def masked_mean(loss,mask):
    mask=mask.to(dtype=loss.dtype)
    return (loss*mask).sum()/mask.sum().clamp(min=1)


def masked_barrier_loss(o,b):
    B=len(b[0]);nh=len(H)
    logits=o['barrier'].float().reshape(-1,3)
    losses=F.cross_entropy(logits,b[16].long().reshape(-1),reduction='none').reshape(B,nh)
    return masked_mean(losses,b[20])


def masked_decision_loss(o,b):
    return masked_mean(F.cross_entropy(o['decision'].float(),b[6].long(),reduction='none'),b[19])


def action_loss(o,b):
    util=cost_aware_utilities(b[1],b[3],b[4]).detach()
    # Utilities supervise action quantiles; hard first-touch labels supervise decision probabilities.
    # Temperature calibration uses exactly these same hard labels.
    # Match the decision head target to temperature calibration and diagnostic reports.
    classification=masked_decision_loss(o,b)
    quantile=pinball(o['action_quantiles']/ret_scale,util/ret_scale)
    return CFG['decision_weight']*(classification+CFG['action_q_weight']*quantile)


def router_balance(o):
    usage=o['dispatch'].mean(0).detach()
    probs=o['router_probs'].mean(0)
    return len(usage)*(usage*probs).sum()


def forecast_losses(o,b,include_regime=True,include_paths=True,include_route=True):
    # Mandatory future outcomes in b are only read for LOSS, never forwarded into the model.
    r,d,m,n,v,decision,pattern,jump,path=b[1],b[2],b[3],b[4],b[5],b[6],b[8],b[9],b[10]
    barrier=b[16]
    total=(CFG['direction_weight']*F.cross_entropy(o['direction'].reshape(-1,3),d.reshape(-1))+
           CFG['quantile_weight']*pinball(o['quantiles']/ret_scale,r/ret_scale)+
           CFG['rv_weight']*F.smooth_l1_loss(torch.log1p(o['rv']/rv_scale),torch.log1p(v/rv_scale))+
           CFG['excursion_weight']*.5*(F.smooth_l1_loss(o['excursions'][...,0]/ret_scale,m/ret_scale)+
             F.smooth_l1_loss(o['excursions'][...,1]/ret_scale,n/ret_scale))+
           CFG['barrier_weight']*masked_barrier_loss(o,b)+
           CFG['pattern_weight']*F.binary_cross_entropy_with_logits(o['pattern'],pattern)+
           CFG['pattern_detail_weight']*F.binary_cross_entropy_with_logits(o['pattern_detail'],b[14])+
           CFG['jump_weight']*F.binary_cross_entropy_with_logits(o['jump'],jump)+
           CFG['path_weight']*F.smooth_l1_loss(o['path'],path))
    if include_paths:
        total=total+CFG['path_nll_weight']*path_distribution_nll(o,b[15])
    if include_regime:
        total=total+CFG['regime_weight']*regime_loss(o,b)
    if include_route:
        total=total+CFG['route_balance_weight']*router_balance(o)
    return total


def objective(o,b):
    return forecast_losses(o,b)+action_loss(o,b)


def joint_pcgrad_backward(o,b):
    # Two-objective PCGrad on shared parameters; private-head gradients remain intact.
    predictive=forecast_losses(o,b);decision_obj=action_loss(o,b)
    params=[p for p in model.parameters() if p.requires_grad]
    a=torch.autograd.grad(predictive,params,retain_graph=True,allow_unused=True)
    g=torch.autograd.grad(decision_obj,params,allow_unused=True)
    common=[i for i,(x,y) in enumerate(zip(a,g)) if x is not None and y is not None]
    if common:
        dot=sum((a[i]*g[i]).sum() for i in common)
        aa=sum(a[i].square().sum() for i in common).clamp(min=1e-12)
        gg=sum(g[i].square().sum() for i in common).clamp(min=1e-12)
        cosine=float((dot/torch.sqrt(aa*gg)).detach())
        conflict=bool(dot.detach().item()<0)
        alpha=(dot/gg).detach() if conflict else None
        beta=(dot/aa).detach() if conflict else None
    else:
        cosine=float('nan');conflict=False;alpha=beta=None
    for i,p in enumerate(params):
        x,y=a[i],g[i]
        if x is not None and y is not None and conflict:
            p.grad=(x-alpha*y+y-beta*x).detach()
        elif x is not None and y is not None:p.grad=(x+y).detach()
        elif x is not None:p.grad=x.detach()
        elif y is not None:p.grad=y.detach()
    return {'gradient_cosine':cosine,'gradient_conflict':conflict}


def select_stage_parameters(stage):
    temporal=('input','conv','encoder','longrange','ssm','graph_fusion','mtf')
    graph=('graph_input','graph','mtf')
    forecast=('mtf','direction','quantiles','rv','excursions','barrier','pattern','pattern_detail','jump','path','forecast_fusion')
    if stage=='A_SSL':prefixes=temporal+('ssl_predict',)
    elif stage=='B_REGIME':prefixes=temporal+('regime',)
    elif stage=='C_GRAPH':prefixes=graph+('graph_fusion','forecast_fusion','direction','quantiles')
    elif stage=='D_FORECAST':prefixes=temporal+graph+('regime',)+forecast
    elif stage=='E_PATH':prefixes=temporal+graph+('forecast_fusion','path_distribution','path')
    elif stage=='F_EXPERTS':prefixes=temporal+graph+('regime','router','experts','path_distribution')+forecast
    elif stage=='G_DECISION':prefixes=('decision','decision_quantiles')
    elif stage=='H_JOINT':prefixes=tuple({s.split('.')[0] for s,_ in model.named_parameters()})
    else:raise ValueError(stage)
    for name,p in model.named_parameters():p.requires_grad=name.split('.')[0] in prefixes
    active=sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert active>0
    return active


STAGE_GRADIENT_HEADS={
    'A_SSL':('ssl_predict',),
    'B_REGIME':('regime',),
    'C_GRAPH':('direction','quantiles'),
    'D_FORECAST':('direction','quantiles','rv','barrier','path'),
    'E_PATH':('path_distribution','path'),
    'F_EXPERTS':('router','direction','path_distribution'),
    'G_DECISION':('decision','decision_quantiles'),
    'H_JOINT':('direction','decision','path_distribution','path'),
}


def assert_stage_head_gradients(stage):
    if not CFG['diagnostic_gradient_assert']:return {}
    diagnostic={}
    for head in STAGE_GRADIENT_HEADS[stage]:
        ps=[p for name,p in model.named_parameters() if name.startswith(head+'.')]
        if not ps:raise RuntimeError(f'{stage}: missing expected module {head}')
        if not any(p.requires_grad for p in ps):raise RuntimeError(f'{stage}: {head} is frozen but its loss requires training')
        grads=[p.grad for p in ps if p.grad is not None]
        if not grads:raise RuntimeError(f'{stage}: missing gradient to {head}')
        if any(not torch.isfinite(g).all().item() for g in grads):raise FloatingPointError(f'{stage}: nonfinite gradient in {head}')
        magnitude=sum(float(g.detach().float().abs().sum()) for g in grads)
        if magnitude<=0:raise RuntimeError(f'{stage}: zero gradient in {head}')
        diagnostic[head]=magnitude
    return diagnostic


def stage_step(stage,b):
    x=b[0]
    if stage=='A_SSL':
        o=model(x[:,:-1],None,None,use_experts=False,use_mtf=False)
        return F.smooth_l1_loss(o['ssl_prediction'],x[:,-1].clamp(-12,12))
    has_experts=stage in ('F_EXPERTS','G_DECISION','H_JOINT')
    graph_on=stage not in ('B_REGIME',)
    o=model(x,b[11] if graph_on else None,b[12] if graph_on else None,use_experts=has_experts,mtf=b[17],mtf_mask=b[18])
    if stage=='B_REGIME':return regime_loss(o,b)
    if stage=='C_GRAPH':
        return F.cross_entropy(o['direction'].reshape(-1,3),b[2].reshape(-1)) + pinball(o['quantiles']/ret_scale,b[1]/ret_scale)
    if stage=='D_FORECAST':return forecast_losses(o,b,include_paths=False,include_route=False)
    if stage=='E_PATH':return path_distribution_nll(o,b[15])+F.smooth_l1_loss(o['path'],b[10])
    if stage=='F_EXPERTS':return forecast_losses(o,b,include_paths=True,include_route=True)
    if stage=='G_DECISION':return action_loss(o,b)
    if stage=='H_JOINT':return objective(o,b)
    raise ValueError(stage)


def atomic_save(obj,path):
    temp=path.with_suffix(path.suffix+'.tmp');torch.save(obj,temp);temp.replace(path)


def rng_snapshot():
    snapshot={'torch':torch.get_rng_state(),'numpy':np.random.get_state(),'python':random.getstate()}
    if torch.cuda.is_available():snapshot['cuda']=torch.cuda.get_rng_state_all()
    return snapshot


def rng_restore(snapshot):
    torch.set_rng_state(snapshot['torch']);np.random.set_state(snapshot['numpy']);random.setstate(snapshot['python'])
    if torch.cuda.is_available() and 'cuda' in snapshot:torch.cuda.set_rng_state_all(snapshot['cuda'])


In [ ]:
# Regression: low-rank Cholesky must run in FP32 under outer BF16 autocast.
n_knots=len(CFG['path_knots'])
probe_outputs={
    'path_mu':(torch.randn(2,n_knots,device=DEVICE)*.001).requires_grad_(),
    'path_factors':(torch.randn(2,n_knots,CFG['path_rank'],device=DEVICE)*.001).requires_grad_(),
    'path_diag':torch.ones(2,n_knots,device=DEVICE,requires_grad=True)*.001
}
with amp_context():
    regression_nll=path_distribution_nll(probe_outputs,torch.randn(2,n_knots,device=DEVICE)*.001)
    regression_paths=sample_correlated_price_paths(probe_outputs,starting_price=100.,scenarios=2)
assert regression_nll.dtype==torch.float32 and bool(torch.isfinite(regression_nll))
assert regression_paths.dtype==torch.float32 and bool(torch.isfinite(regression_paths).all())
regression_nll.backward()
for name in ('path_mu','path_factors'):
    assert probe_outputs[name].grad is not None
    assert bool(torch.isfinite(probe_outputs[name].grad).all())
print('L4 BF16 -> FP32 LOW-RANK CHOLESKY / BACKWARD REGRESSION: PASS')

# 4 — Parameter & stage contract checks
assert PARAMETERS==10041978, f'Model count changed: {PARAMETERS}'
STAGES=list(CFG['stage_epochs'])
assert STAGES==['A_SSL','B_REGIME','C_GRAPH','D_FORECAST','E_PATH','F_EXPERTS','G_DECISION','H_JOINT']
assert set(STAGES)==set(CFG['finetune_epochs'])
assert CFG['pool_assets']==['BTCUSD','ETHUSDT','SOLUSDT'] and CFG['target_asset']=='BTCUSD'
assert CFG['use_pcgrad_joint']
print('Full architecture + stage configuration match the trainer')


In [ ]:
# 5 — Synthetic batch matching the production 21-element LazyFold tuple
Fdim=len(FEATURES);nh=len(H);hp=CFG['graph_history_points']
K=len(TF_NAMES); W=CFG['mtf_window']; neighbor_count=3

def synthetic_batch(B=DIAGNOSTIC_BATCH_SIZE,seed=None):
    if seed is not None: torch.manual_seed(seed)
    dev=DEVICE
    rand=lambda *shape:torch.randn(*shape,device=dev,dtype=torch.float32)
    x=(rand(B,lb,Fdim)*.1).clamp(-2,2)
    future_ret=rand(B,nh)*.002
    direction=torch.randint(0,3,(B,nh),device=dev)
    mfe=rand(B,nh).abs()*.003
    mae=-rand(B,nh).abs()*.003
    rv=rand(B,nh).abs()*.002
    decision=torch.randint(0,3,(B,),device=dev)
    regime=torch.rand(B,7,device=dev)
    pattern=torch.rand(B,nh,device=dev)
    jump=torch.rand(B,nh,device=dev)
    path=rand(B,nh)*.002
    neighbors=rand(B,neighbor_count,hp,Fdim)*.05
    neighbor_mask=torch.ones(B,neighbor_count,hp,dtype=torch.bool,device=dev)
    regime_mask=torch.ones(B,7,device=dev)
    pattern_detail=torch.rand(B,nh,3,device=dev)
    increments=rand(B,len(CFG['path_knots']))*.001
    barriers=torch.randint(0,3,(B,nh),device=dev)
    mtf=rand(B,K,W,Fdim)*.05
    mtf_mask=torch.ones(B,K,W,dtype=torch.bool,device=dev)
    decision_mask=torch.ones(B,dtype=torch.bool,device=dev)
    barrier_mask=torch.ones(B,nh,dtype=torch.bool,device=dev)
    if B>1:decision_mask[1]=False
    barrier_mask[0,0]=False
    return (x,future_ret,direction,mfe,mae,rv,decision,regime,pattern,jump,path,neighbors,neighbor_mask,
            regime_mask,pattern_detail,increments,barriers,mtf,mtf_mask,decision_mask,barrier_mask)

batch=synthetic_batch()
assert len(batch)==21 and batch[0].shape==(DIAGNOSTIC_BATCH_SIZE,lb,Fdim)
assert batch[11].shape==(DIAGNOSTIC_BATCH_SIZE,3,hp,Fdim)
assert batch[17].shape==(DIAGNOSTIC_BATCH_SIZE,K,W,Fdim)
print('Tensor shapes and target masks:',[tuple(v.shape) for v in batch])


In [ ]:
# 5b — Bounded read-only REAL parquet probe of production features and labels.
# This is an added data-quality gate, NOT a substitute for the complete LazyFold DataLoader.
REAL_DATA_PROBES={}
if DATA_READY and pq is not None:
    for asset,path in DATA_PATHS.items():
        rec={'asset':asset,'path':str(path),'available':True}
        try:
            pf=pq.ParquetFile(path)
            last=pf.metadata.num_row_groups-1
            # Refuse unbounded/giant row groups in diagnostic runs.
            selected=[];nrows=0
            for idx in range(last,max(-1,last-4),-1):
                rg_rows=pf.metadata.row_group(idx).num_rows
                if rg_rows>250000:raise RuntimeError('Row group too large for bounded smoke test')
                selected.append(idx);nrows+=rg_rows
                if nrows>=1200:break
            if nrows<CFG['min_rows']:raise RuntimeError('Too few recent rows for source feature probe')
            selected=list(reversed(selected))
            chunk=pf.read_row_groups(selected).to_pandas()
            chunk=chunk.iloc[-max(1200,CFG['min_rows']+500):].copy()
            with tempfile.TemporaryDirectory(prefix='ananta_data_probe_') as tmp:
                path_tmp=Path(tmp)/f'{asset}_M5.csv'
                chunk=chunk.reset_index() if isinstance(chunk.index,pd.DatetimeIndex) else chunk
                if 'timestamp' not in chunk.columns:
                    for key in ('datetime','date_time','time','date','index'):
                        if key in chunk.columns:chunk=chunk.rename(columns={key:'timestamp'});break
                chunk.to_csv(path_tmp,index=False)
                d=load_asset_file(path_tmp)
                feature_df=causal_features(d)
                label_df=build_labels(d,H)
            assert set(FEATURES).issubset(feature_df.columns)
            assert all(f'ret_{h}' in label_df.columns for h in H)
            valid_feat=int(np.isfinite(feature_df[FEATURES].to_numpy(dtype=float)).all(1).sum())
            label_valid=int(np.isfinite(label_df[f'ret_{max(H)}'].to_numpy(dtype=float)).sum())
            if asset in CFG['pool_assets'] and (valid_feat<32 or label_valid<32):
                raise RuntimeError('Insufficient valid feature/label rows in cached sample')
            rec.update({'passed':True,'sample_rows':len(d),'valid_feature_rows':valid_feat,
                        'valid_horizon_rows':label_valid})
        except Exception as e:
            rec.update({'passed':False,'error':str(e)})
        REAL_DATA_PROBES[asset]=rec
        print('REAL FEATURE/LABEL PROBE',asset,rec)
else:
    print('REAL feature/label probes skipped: cached files not ready. Data gate remains open.')
REAL_PROBE_OK=DATA_READY and all(v['passed'] for v in REAL_DATA_PROBES.values()) and len(REAL_DATA_PROBES)==4


In [ ]:
# 6 — GPU smoke: exact production stage_step / objective / PCGrad + AdamW + validation.
# One step per stage is deliberately NOT a full-epoch benchmark.
records=[]
if DEVICE.type=='cuda':torch.cuda.reset_peak_memory_stats()
for phase in ('pooled','btc_finetune'):
    for stage in STAGES:
        model.train()
        active=select_stage_parameters(stage)
        lr=CFG['lr']*CFG['stage_lr_mult'][stage]*(CFG['finetune_lr_multiplier'] if phase=='btc_finetune' else 1.)
        opt=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=lr,weight_decay=.01)
        times=[];losses=[];diag={};grad_norm=float('nan')
        for t in range(TEST_STEPS_PER_STAGE):
            b=synthetic_batch(seed=2026+t+37*STAGES.index(stage))
            opt.zero_grad(set_to_none=True)
            if DEVICE.type=='cuda':torch.cuda.synchronize()
            ts=time.perf_counter()
            with amp_context():
                if stage=='H_JOINT' and CFG['use_pcgrad_joint']:
                    out=model(b[0],b[11],b[12],mtf=b[17],mtf_mask=b[18])
                    loss=objective(out,b)
                else:
                    loss=stage_step(stage,b)
            assert bool(torch.isfinite(loss).item()),f'{phase}/{stage}: nonfinite loss'
            if stage=='H_JOINT' and CFG['use_pcgrad_joint']:
                with amp_context():pcdiag=joint_pcgrad_backward(out,b)
            else:loss.backward()
            diag=assert_stage_head_gradients(stage)
            norms=[g for p in model.parameters() if (g:=p.grad) is not None]
            assert norms and all(bool(torch.isfinite(g).all().item()) for g in norms), f'{phase}/{stage}: invalid gradient'
            gn=nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.)
            assert bool(torch.isfinite(gn).item())
            grad_norm=float(gn.detach());opt.step()
            if DEVICE.type=='cuda':torch.cuda.synchronize()
            times.append(time.perf_counter()-ts)
            losses.append(float(loss.detach()))
        model.eval()
        with torch.inference_mode(),amp_context():
            valid_loss=stage_step(stage,synthetic_batch(seed=9000+STAGES.index(stage)))
        assert bool(torch.isfinite(valid_loss).item())
        result={'phase':phase,'stage':stage,'trainable_parameters':int(active),
            'train_steps':TEST_STEPS_PER_STAGE,'mean_train_loss':float(np.mean(losses)),
            'synthetic_val_loss':float(valid_loss), 'median_step_s':float(np.median(times)),
            'gradient_norm_preclip':grad_norm,'required_head_gradient_l1':diag}
        records.append(result)
        print(phase,stage,'OK',round(result['median_step_s'],3),'sec/step','val_loss',round(result['synthetic_val_loss'],5))
        del opt
        gc.collect()
assert len(records)==2*len(STAGES)


In [ ]:
# 7 — isolated checkpoint / optimizer / sampler RNG / resume contract round-trip
model.train();select_stage_parameters('G_DECISION')
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-5,weight_decay=.01)
b=synthetic_batch(B=2,seed=1001)
opt.zero_grad(set_to_none=True)
with amp_context(): loss=stage_step('G_DECISION',b)
loss.backward();assert_stage_head_gradients('G_DECISION');opt.step()
rng=rng_snapshot()
with tempfile.TemporaryDirectory(prefix='ananta_preflight_') as d:
    path=Path(d)/'isolated_resume.pt'
    snap={'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
          'optimizer':opt.state_dict(),'rng':rng,'phase':'btc_finetune',
          'stage':'G_DECISION','epoch':1,'batch_offset':250}
    atomic_save(snap,path)
    loaded=torch.load(path,map_location='cpu',weights_only=False)
    assert loaded['batch_offset']==250 and loaded['phase']=='btc_finetune'
    assert all(torch.equal(v,loaded['model'][k]) for k,v in snap['model'].items())
    restore_opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-5,weight_decay=.01)
    restore_opt.load_state_dict(loaded['optimizer'])
    assert restore_opt.state_dict()['state']
    rng_restore(loaded['rng'])
print('ISOLATED checkpoint/state roundtrip PASS (not a Colab interruption/restart test)')


In [ ]:
# 8 — report. Never claim READY FOR FULL TRAINING from synthetic tests alone.
now=datetime.now(timezone.utc).isoformat()
VRAM={'allocated_gib':float(torch.cuda.max_memory_allocated()/2**30),
      'reserved_gib':float(torch.cuda.max_memory_reserved()/2**30)} if DEVICE.type=='cuda' else None
report={'utc':now,'notebook':'A-nanTa V7.3.1.4 matched preflight',
   'source_url':MAIN_RAW_URL,'source_hashes':EXPECTED_SOURCE_HASHES,'provenance':PROVENANCE_STATUS,
   'device':str(DEVICE),'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
   'bf16':bool(USE_BF16),'full_model_parameters':int(PARAMETERS),
   'data_audit':DATA_AUDIT,'data_ready':bool(DATA_READY), 'real_data_probes':REAL_DATA_PROBES,'real_probe_ok':bool(REAL_PROBE_OK),
   'stages':records,'vram':VRAM,'checkpoint_roundtrip':True,
   'real_data_training_tested':False,
   'training_ready':False,
   'verdict':'MODEL_KERNEL_PASS_DATA_UNVERIFIED' if not DATA_READY else 'MODEL_KERNEL_PASS_COVERAGE_NOT_YET_PROVEN',
   'notes':'Does not test real DataLoader or full 5-year coverage; real-data L4 gate remains required.'}
out=REPORT_DIR/'A-nanTa_V7314_preflight_report.json'
out.write_text(json.dumps(report,indent=2,default=str))
print('PRE-FLIGHT MODEL KERNEL: PASS')
print('DATA CACHES:', 'PRESENT' if DATA_READY else 'MISSING/INVALID')
print('TRAINING READY:',report['training_ready'],'(real-data end-to-end still required)')
print('REPORT:',out)
